# CASMI 2026 — spectrum → fingerprint model (training)

Trains an MLP that maps a binned MS/MS spectrum (fragments + neutral losses + precursor/adduct scalars) to a 2048-bit Morgan (r=2) fingerprint of the molecule. Used downstream to score **candidate structures that have no library spectra** (novelty class 2): candidates come from a mass shortlist (natural-product table + training structures), ranked by expected Tanimoto between predicted bit probabilities and the candidate's fingerprint.

Validation is by *structure* (held-out structures are never in training) so it mimics class 2. Outputs `fpmodel.pt` (weights + config) in the working dir; the inference notebook consumes it as a kernel source.

In [ ]:
import os, glob, time, re
import numpy as np, pandas as pd, pyarrow.parquet as pq, pyarrow as pa, pyarrow.compute as pc
import scipy.sparse as sp, torch, torch.nn as nn
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator
RDLogger.DisableLog("rdApp.*")
DATA_DIR = os.environ.get("CASMI_DIR") or next(iter(glob.glob("/kaggle/input/**/train.parquet", recursive=True)), None)
DATA_DIR = os.path.dirname(DATA_DIR) if DATA_DIR.endswith(".parquet") else DATA_DIR
OUT_DIR = os.environ.get("CASMI_OUT", "/kaggle/working" if os.path.isdir("/kaggle/working") else ".")
FP_BITS = 2048
PER_STRUCT = int(os.environ.get("PER_STRUCT", 4)); EPOCHS = int(os.environ.get("EPOCHS", 8)); N_STRUCT = int(os.environ.get("N_STRUCT", 0))
dev = "cuda" if torch.cuda.is_available() else "cpu"; print(DATA_DIR, dev)

In [ ]:
# shared featurisation (pasted into both notebooks)
import numpy as np, scipy.sparse as sp
FRAG_BINS, LOSS_BINS = 1000, 500
ADDUCTS = ["[M+H]+","[M+NH4]+","[M-H2O+H]+","[M-2H2O+H]+","[M+Na]+","[M+K]+","[M-H]-","[M-H2O-H]-","[M+CH2O2-H]-","[M+Cl]-"]
N_SCALAR = len(ADDUCTS) + 3
IN_DIM = FRAG_BINS + LOSS_BINS + N_SCALAR
def featurize(mzs, ints, prec, adduct, ion):
    """-> CSR (n, IN_DIM); invalid spectra give empty rows."""
    r, c, v = [], [], []
    n = len(mzs)
    for i in range(n):
        mz = np.asarray(mzs[i], float); it = np.asarray(ints[i], float); p = float(prec[i])
        if len(mz) and it.max() > 0:
            it = it / it.max()
            m = (mz <= p + 2) & (it >= 0.005) & (mz > 0)
            mz, it = mz[m], np.sqrt(it[m])
            if len(mz) > 150:
                t = np.argsort(-it)[:150]; mz, it = mz[t], it[t]
            if len(mz):
                it = it / np.linalg.norm(it)
                fb = np.minimum(mz.astype(int), FRAG_BINS - 1)
                loss = p - mz; lm = (loss >= 0) & (loss < LOSS_BINS)
                r.append(np.full(len(fb), i)); c.append(fb); v.append(it)
                r.append(np.full(lm.sum(), i)); c.append(FRAG_BINS + loss[lm].astype(int)); v.append(it[lm])
        ai = ADDUCTS.index(adduct[i]) if adduct[i] in ADDUCTS else None
        base = FRAG_BINS + LOSS_BINS
        r.append(np.array([i, i, i])); c.append(np.array([base + len(ADDUCTS), base + len(ADDUCTS) + 1, base + len(ADDUCTS) + 2]))
        v.append(np.array([min(p, 1500) / 1000.0, 1.0 if ion[i] == "positive" else 0.0, 1.0 if ai is None else 0.0]))
        if ai is not None:
            r.append(np.array([i])); c.append(np.array([base + ai])); v.append(np.array([1.0]))
    X = sp.csr_matrix((np.concatenate(v), (np.concatenate(r), np.concatenate(c))), shape=(n, IN_DIM), dtype=np.float32)
    return X


In [ ]:
# ---- fingerprints per structure ----
_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=FP_BITS)
def smiles_fp(smi):
    m = Chem.MolFromSmiles(smi)
    return None if m is None else np.packbits(_gen.GetFingerprintAsNumPy(m).astype(np.uint8))
meta = pq.read_table(f"{DATA_DIR}/train.parquet", columns=["inchikey14","normalized_smiles","ingest_lib"]).to_pandas()
meta["row"] = np.arange(len(meta))
structs = meta.drop_duplicates("inchikey14")[["inchikey14","normalized_smiles"]].reset_index(drop=True)
if N_STRUCT: structs = structs.sample(N_STRUCT, random_state=0).reset_index(drop=True)
t0 = time.time()
fps = [smiles_fp(s) for s in structs.normalized_smiles]
ok = np.array([f is not None for f in fps]); structs = structs[ok].reset_index(drop=True)
FP = np.stack([f for f in fps if f is not None]); print(FP.shape, f"{time.time()-t0:.0f}s")
sidx = {k: i for i, k in enumerate(structs.inchikey14)}

In [ ]:
# ---- choose spectra: up to PER_STRUCT per structure; hold out 2% of structures ----
rng = np.random.RandomState(0)
meta = meta[meta.inchikey14.isin(sidx)].copy()
meta["sid"] = meta.inchikey14.map(sidx)
is_val = rng.rand(len(structs)) < 0.02
meta["is_val"] = is_val[meta.sid.to_numpy()]
# prefer timsTOF/enveda-np rows first, then random
meta["pri"] = rng.rand(len(meta)) - meta.ingest_lib.isin(["enveda-np-examples"]).astype(float)
sel = meta.sort_values("pri").groupby("sid").head(PER_STRUCT).sort_values("row")
want = sel.set_index("row")["sid"].to_dict(); want_val = sel.set_index("row")["is_val"].to_dict()
print(len(sel), "spectra selected;", int(is_val.sum()), "val structures")

In [ ]:
# ---- featurize selected spectra (stream the file) ----
cols = ["ms2_mzs","ms2_normalized_intensities","precursor_mz","adduct","ionization_mode"]
pf = pq.ParquetFile(f"{DATA_DIR}/train.parquet"); off = 0
Xs, S, V = [], [], []; t0 = time.time()
wrows = np.array(sorted(want))
for b in pf.iter_batches(batch_size=50000, columns=cols):
    n = b.num_rows; lo, hi = np.searchsorted(wrows, [off, off + n]); idx = wrows[lo:hi] - off
    if len(idx):
        d = pa.Table.from_batches([b]).take(pa.array(idx)).to_pandas()
        Xs.append(featurize(d.ms2_mzs.tolist(), d.ms2_normalized_intensities.tolist(), d.precursor_mz.to_numpy(), d.adduct.tolist(), d.ionization_mode.tolist()))
        gr = wrows[lo:hi]; S.append([want[r] for r in gr]); V.append([want_val[r] for r in gr])
    off += n
X = sp.vstack(Xs).tocsr(); S = np.concatenate(S); V = np.concatenate(V)
print(X.shape, f"{time.time()-t0:.0f}s")

In [ ]:
# ---- model ----
class Net(nn.Module):
    def __init__(s, d_in=IN_DIM, h=2048, out=FP_BITS, p=0.2):
        super().__init__()
        s.f = nn.Sequential(nn.Linear(d_in, h), nn.ReLU(), nn.Dropout(p), nn.Linear(h, h), nn.ReLU(), nn.Dropout(p), nn.Linear(h, out))
    def forward(s, x): return s.f(x)
def to_dense(Xb): return torch.from_numpy(Xb.toarray()).to(dev)
def unpack(idx): return torch.from_numpy(np.unpackbits(FP[idx], axis=1).astype(np.float32)).to(dev)

tr = np.flatnonzero(~V); va = np.flatnonzero(V)
net = Net().to(dev); opt = torch.optim.AdamW(net.parameters(), 1e-3, weight_decay=1e-4)
BS = 512; steps = EPOCHS * (len(tr) // BS + 1)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=steps)
lossf = nn.BCEWithLogitsLoss()
def predict_probs(Xm, bs=2048):
    net.eval(); out = []
    with torch.no_grad():
        for i in range(0, Xm.shape[0], bs): out.append(torch.sigmoid(net(to_dense(Xm[i:i+bs]))).cpu().numpy())
    net.train(); return np.vstack(out)
for ep in range(EPOCHS):
    perm = np.random.permutation(tr); tl = 0; t0 = time.time()
    for i in range(0, len(perm), BS):
        b = np.sort(perm[i:i+BS]); xb = to_dense(X[b]); yb = unpack(S[b])
        loss = lossf(net(xb), yb); opt.zero_grad(); loss.backward(); opt.step(); sched.step(); tl += loss.item() * len(b)
    vl = 0
    if len(va):
        P = predict_probs(X[va[:20000]]); y = np.unpackbits(FP[S[va[:20000]]], axis=1)
        vl = float(-(y*np.log(P+1e-7)+(1-y)*np.log(1-P+1e-7)).mean())
    print(f"epoch {ep}: train {tl/len(perm):.4f} val {vl:.4f} ({time.time()-t0:.0f}s)")

In [ ]:
# ---- structure-level retrieval validation (class-2 proxy) ----
def formula_mass(f):
    EL = {"H":1.00782503207,"C":12.0,"N":14.0030740048,"O":15.99491461956,"F":18.99840322,"Na":22.9897692809,"P":30.97376163,"S":31.972071,"Cl":34.96885268,"K":38.96370668,"Br":78.9183371,"I":126.904473,"B":11.0093054,"Si":27.9769265325,"Se":79.9165213}
    m = 0.0
    for el, n in re.findall(r"([A-Z][a-z]?)(\d*)", f):
        if el not in EL: return np.nan
        m += EL[el] * (int(n) if n else 1)
    return m
def expected_tanimoto(p, F):            # p: (bits,), F: (n, bits) binary
    inter = F @ p; return inter / (p.sum() + F.sum(1) - inter + 1e-9)
if len(va):
    fm = pq.read_table(f"{DATA_DIR}/train.parquet", columns=["molecular_formula","inchikey14"]).to_pandas().drop_duplicates("inchikey14")
    fm = fm[fm.inchikey14.isin(sidx)]; fm["sid"] = fm.inchikey14.map(sidx); fm["mass"] = fm.molecular_formula.map(formula_mass)
    mass = np.full(len(structs), np.nan); mass[fm.sid.to_numpy()] = fm.mass.to_numpy()
    order = np.argsort(np.nan_to_num(mass, nan=1e9)); sm = mass[order]
    Pv = predict_probs(X[va]); rr = []
    vs = np.unique(S[va])[:500]
    for s in vs:
        if np.isnan(mass[s]): continue
        p = Pv[S[va] == s].mean(0)
        lo, hi = np.searchsorted(sm, [mass[s] - 0.01, mass[s] + 0.01]); cand = order[lo:hi]
        F = np.unpackbits(FP[cand], axis=1).astype(np.float32)
        sc = expected_tanimoto(p, F); rank = (sc > sc[list(cand).index(s)]).sum() + 1
        rr.append(1.0 / rank if rank <= 25 else 0.0)
    print(f"held-out-structure MRR@25 within +-0.01 Da pool of all train structures: {np.mean(rr):.3f} (n={len(rr)}, median cands ~{hi-lo})")

In [ ]:
torch.save({"state": net.state_dict(), "in_dim": IN_DIM, "fp_bits": FP_BITS, "hidden": 2048}, f"{OUT_DIR}/fpmodel.pt")
print("saved", os.path.getsize(f"{OUT_DIR}/fpmodel.pt") / 1e6, "MB")